Raw Emails
 → Clean text
 → Tokenize
 → TF-IDF
 → Train model
 → Evaluate
 → Predict new emails

In [1]:
import pandas as pd

df = pd.read_csv("data/enron_spam_data.csv", encoding="latin-1")

df.head()

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Message ID  33716 non-null  int64 
 1   Subject     33427 non-null  object
 2   Message     33345 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [3]:
df["Spam/Ham"].value_counts()

Spam/Ham
spam    17171
ham     16545
Name: count, dtype: int64

In [4]:
df["text"] = df["Subject"].fillna("") + " " + df["Message"].fillna("")

In [5]:
df[["text", "Spam/Ham"]].head()

,text,Spam/Ham
0,christmas tree farm pictures,ham
1,"vastar resources , inc . gary , production fro...",ham
2,calpine daily gas nomination - calpine daily g...,ham
3,re : issue fyi - see note below - already done...,ham
4,meter 7268 nov allocation fyi .\n- - - - - - -...,ham


In [6]:
df["label"] = df["Spam/Ham"].map({"ham": 0, "spam": 1})

In [7]:
df[["text", "label"]].head()

,text,label
0,christmas tree farm pictures,0
1,"vastar resources , inc . gary , production fro...",0
2,calpine daily gas nomination - calpine daily g...,0
3,re : issue fyi - see note below - already done...,0
4,meter 7268 nov allocation fyi .\n- - - - - - -...,0


We will: 
- Lowercase
- Remove punctuation
- Remove numbers
- Remove stopwords
- Normalize whitespace

In [8]:
import re 
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text): 
    text = text.lower() #lowercase
    text = re.sub(r'\d+', '', text) # remove numbers
    text = re.sub(r'[^\w\s]', '', text) # remove punctuation
    text = text.split() #tokenize
    text = [word for word in text if word not in ENGLISH_STOP_WORDS] # remove stopwords
    return " ".join(text)

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)

In [9]:
df[["text", "clean_text"]].head(5)

,text,clean_text
0,christmas tree farm pictures,christmas tree farm pictures
1,"vastar resources , inc . gary , production fro...",vastar resources gary production high island l...
2,calpine daily gas nomination - calpine daily g...,calpine daily gas nomination calpine daily gas...
3,re : issue fyi - see note below - already done...,issue fyi note stella forwarded stella l morri...
4,meter 7268 nov allocation fyi .\n- - - - - - -...,meter nov allocation fyi forwarded lauri allen...


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df["clean_text"])

y = df["label"]

In [11]:
X.shape

(33716, 5000)

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [14]:
y_pred = model.predict(X_test)

In [15]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy:  0.9823546856465006
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      3276
           1       0.98      0.99      0.98      3468

    accuracy                           0.98      6744
   macro avg       0.98      0.98      0.98      6744
weighted avg       0.98      0.98      0.98      6744



In [16]:
def predict_spam(email_text): 
    #cleaning of raw email
    cleaned = clean_text(email_text)

    #convert to TF-IDF vector 
    vector = vectorizer.transform([cleaned])

    #Predict
    prediction = model.predict(vector)[0]

    #Result
    if prediction == 1:
        return "SPAM"
    else: 
        return "HAM (Not SPAM)"

In [ ]:
email1 = input("Enter an email: ")
email2 = input("Enter an email: ")
print(predict_spam(email1))
print(predict_spam(email2))

Enter an email:  Congratulations Jack! You have just won $10000 and a Tesla. Click here below right now to claim!


In [ ]:
Congratulations Jack! You have just won $10000 and a Tesla. Click here below right now to claim! Hey Jack, there is meeting tomorrow at 2pm to discuss about the project.